# Accuracy and the questions that relate the views

S-Mind's rows of Table 1 on the four benchmarks, after SFT and after RL, and the accuracy of S-Mind after SFT on the
questions that relate the views (Fig. 8b): the two-view questions of MindCube-Tiny, which ask how the camera moved
between two photos, and the multi-view reasoning questions of BLINK.

**Needs:** the four benchmarks built (`scripts/data/benchmarks.sh`); one GPU, about an hour per model on one H100.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import smind_nb as nb
nb.setup()

SIZE = "4b"   # or "9b"
GPU = "0"

In [ ]:
SETS = ["tiny", "sparbench", "blink_spatial", "viewspatial"]
NAMES = ["MindCube-Tiny", "SPAR-Bench (R&V)", "BLINK (Spatial)", "ViewSpatial"]
nb.need(*[f"data/processed/{f}" for f in ("mindcube_tiny.jsonl", "bench_sparbench.jsonl", "bench_blink_spatial.jsonl",
                                          "bench_viewspatial.jsonl")])
score = {}
for stage in ("sft", "rl"):
    ckpt, out = nb.checkpoint(SIZE, stage), nb.RUNS / f"{SIZE}-{stage}/eval"
    nb.run(["python", "-m", "smind.eval.generate_ours", "--ckpt", ckpt, "--out", out, "--gpu_mem", "0.8"],
           out / "generate.log", done=out / "viewspatial/summary.json", gpu=GPU)
    nb.run(["python", "-m", "smind.eval.summarize", "--out", out], out / "summarize.log",
           done=out / "viewspatial/summary.json")
    s = {n: nb.read_json(out / f"{b}/summary.json") for b, n in zip(SETS, NAMES)}
    score[stage] = {n: 100 * (v["acc_task_mean"] if b == "sparbench" else v["acc"]) for (b, n), v in zip(zip(SETS, NAMES), s.values())}
    score[stage]["Average"] = sum(score[stage].values()) / 4
import pandas as pd
df = pd.DataFrame({f"S-Mind ({s.upper()})": v for s, v in score.items()})
if SIZE == "4b":
    df["paper: SFT"] = [82.29, 64.24, 75.00, 46.17, 66.92]
    df["paper: RL"] = [88.00, 71.03, 78.00, 54.27, 72.83]
display(df)

In [ ]:
out = nb.RUNS / f"{SIZE}-sft"
nb.run(["python", "analysis/diagnosis/two_view_errors.py", "--model", f"sft={out / 'eval'}", "--out",
        out / "two_view.json"], out / "two_view.log", done=out / "two_view.json")
two = nb.read_json(out / "two_view.json")["two_view"]["sft"]
blink = [p for p in nb.jsonl(out / "eval/blink_spatial/predictions.jsonl") if p["task"] == "Multi-view_Reasoning"]
rows = {f"MindCube-Tiny, two-view questions (n={two['n']})": two["acc"][0],
        f"BLINK, multi-view reasoning (n={len(blink)})": 100 * sum(p["correct"] in (True, "True") for p in blink) / len(blink)}
nb.compare(rows, dict(zip(rows, [96.35, 59.40])) if SIZE == "4b" else None)

On the two-view questions, which hinge on placing the second camera relative to the first, S-Mind after SFT is right
on nearly all of them; Qwen3.5 (Base) answers them near chance (`06_base_diagnosis.ipynb`).